## Extract Habitat Metrics by CCFRP Grid Cells
The following code outlines how to generate a dataframe that calculates the average of each habitat metric created by xDEM within each grid cell sampled by CCFRP. The resultant dataframe includes metadata from the grid cell shapefile that describes the grid cell area, unique ID, and whether it is in a MPA or reference site. 

To do this, we need to download the rasterstats package, which is dependent on outdated version of numpy. I achieve this by launching this notebook from my Anaconda Prompt in an environment that supports rasterstats using the depreciated numpy package. 

In [4]:
import geopandas as gpd
import pandas as pd
import rasterio
import os
import re
from rasterstats import zonal_stats

ModuleNotFoundError: No module named 'numpy.distutils'

First, let's connect to the locations of the data we are interested in using and define the data that we want to retain from the grid cell shapefile that we are using in this zonal statistics opeartion. 

In [12]:
# --- CONFIGURATION ---

raster_folder = "xDEM_Rasters"


# Metadata fields from shapefile to retain
area_full_field = "Full_Area"
area_code_field = "Area"               
grid_cell_id_field = "Grid_Cell_"     
mpa_field = "Site__MPA_"


Now, let's slightly edit convention of the area names found in the grid cell shapefile (e.g. "Ano Nuevo") so that they match the naming conventions of the xDEM habitat rasters (e.g. "AnoNuevo"). 

In [5]:
grid_cells_path = "Shapefiles/CCFRP_Grid_Cells_reprojected/CCFRP_Grid_Cells.shp"

grid_cells = gpd.read_file(grid_cells_path)

grid_cells["Compressed_Area"] = grid_cells['Full_Area'].str.replace(" ", "")
grid_cells

,TARGET_FID,Grid_Cell_,Area,Site__MPA_,Old_New,Lat_Center,Lon_Center,Lat_1,Lon_1,Lat_2,...,Cell_Sa_12,Cell_Sa_13,Cell_Sa_14,Shape_Leng,Shape_Area,NEAR_FID,NEAR_DIST,Full_Area,geometry,Compressed_Area
0,1,LJ20,LJ,MPA,None,32.801750,-117.290282,32.799557,-117.293023,32.804070,...,TRUE,TRUE,TRUE,2006.265423,251568.321742,6,23.653079,South La Jolla,"POLYGON ((1.04e+06 3.64e+06, 1.03e+06 3.64e+06...",SouthLaJolla
1,2,LJ19,LJ,MPA,None,32.803349,-117.295917,32.801159,-117.298657,32.805672,...,FALSE,TRUE,TRUE,2006.256371,251566.069192,6,24.139097,South La Jolla,"POLYGON ((1.03e+06 3.64e+06, 1.03e+06 3.64e+06...",SouthLaJolla
2,3,LJ18,LJ,MPA,None,32.805080,-117.302808,32.802891,-117.305549,32.807404,...,TRUE,TRUE,TRUE,2006.282234,251572.559283,6,24.719558,South La Jolla,"POLYGON ((1.03e+06 3.64e+06, 1.03e+06 3.64e+06...",SouthLaJolla
3,4,LJ15,LJ,MPA,None,32.808235,-117.308307,32.806042,-117.311047,32.810555,...,FALSE,TRUE,FALSE,2006.307525,251578.910382,6,25.328745,South La Jolla,"POLYGON ((1.03e+06 3.64e+06, 1.03e+06 3.64e+06...",SouthLaJolla
4,5,LJ17,LJ,MPA,None,32.808346,-117.295828,32.806152,-117.298569,32.810665,...,TRUE,TRUE,FALSE,2006.244091,251562.969929,6,24.547924,South La Jolla,"POLYGON ((1.03e+06 3.64e+06, 1.03e+06 3.64e+06...",SouthLaJolla
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,429,TD07,TD,REF,None,41.114643,-124.173055,41.116924,-124.170117,41.112423,...,TRUE,FALSE,FALSE,1999.619404,249904.849705,5,40.345264,Trinidad,"POLYGON ((4.02e+05 4.55e+06, 4.01e+05 4.55e+06...",Trinidad
429,430,TD08,TD,REF,None,41.121178,-124.171993,41.118958,-124.168975,41.118896,...,TRUE,FALSE,FALSE,2000.468405,250117.100539,5,41.075247,Trinidad,"POLYGON ((4.02e+05 4.55e+06, 4.01e+05 4.55e+06...",Trinidad
430,431,TD09,TD,REF,None,41.121193,-124.179873,41.123474,-124.176936,41.118973,...,TRUE,FALSE,FALSE,1999.608800,249902.200870,5,41.035274,Trinidad,"POLYGON ((4.01e+05 4.55e+06, 4.01e+05 4.55e+06...",Trinidad
431,432,TD10,TD,REF,None,41.130901,-124.176958,41.133183,-124.174020,41.128677,...,TRUE,FALSE,FALSE,2000.468001,250117.000210,5,42.124948,Trinidad,"POLYGON ((4.01e+05 4.55e+06, 4.01e+05 4.55e+06...",Trinidad


Then, we can loop through our folder of xDEM rasters based off of where they match with the areas in the grid cells. First, let's identify the folder that we want to source our rasters from. 

In [14]:
raster_folder = 'xDEM_Rasters'

In [15]:
metadata_fields = ["Full_Area", "Area", "Grid_Cell_", "Site__MPA_"]
all_stats = []

# --- PROCESS EACH RASTER ---
for raster_file in os.listdir(raster_folder):
    if not raster_file.endswith(".tif"):
        continue

    raster_path = os.path.join(raster_folder, raster_file)
    parts = raster_file.replace(".tif", "").split("_")
    raster_area = parts[0]
    metric = "_".join(parts[1:])  # e.g. 'slope', 'rugosity', etc.

    # Match grid cells within area
    matching = grid_cells[grid_cells["Compressed_Area"] == raster_area]
    if matching.empty:
        print(f"⚠️ No match for {raster_area} in shapefile.")
        continue

    # Compute zonal stats
    stats = zonal_stats(
        matching,
        raster_path,
        stats=["mean", "median", "min", "max", "std"],
        nodata=-9999,
        geojson_out=False
    )

    stats_df = pd.DataFrame(stats)

    # Add metric name to each stat column
    stats_df = stats_df.add_prefix(f"{metric}_")  # e.g. slope_mean, slope_max, etc.

    # Keep only metadata
    meta_df = matching[metadata_fields].reset_index(drop=True)

    # Combine metadata + metric stats
    combined_df = pd.concat([meta_df, stats_df], axis=1)

    all_stats.append(combined_df)

# Combine all into one final DataFrame
final_df = pd.concat(all_stats, ignore_index=True)

# Optional: Rename metadata columns for clarity
final_df = final_df.rename(columns={
    "Grid_Cell_": "Grid_Cell_ID",
    "Site__MPA_": "MPA_Status"
})

# Optional: Drop duplicates (some grid cells may show up multiple times if rasters overlap)
final_df = final_df.groupby("Grid_Cell_ID").first().reset_index()

# Save the wide-format output
final_df.to_csv("habitat_metrics_summary_wide.csv", index=False)


C:\Users\FELAB\AppData\Local\Temp\ipykernel_20064\2243037857.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(all_stats, ignore_index=True)
